# Optimale medewerkerroutes met Google OR-Tools

Dit notebook lost een **Vehicle Routing Problem (VRP)** op:
- **20 voertuigen** (medewerkers), elk vertrekkend vanuit hun eigen thuisadres
- **100 cliënten** die verdeeld en bezocht moeten worden
- **Precies 5 stops** per medewerker
- **Minimale totale reistijd** over alle routes gecombineerd

Aanpak:
1. Wegennet laden → NetworkX graph
2. Medewerkers + cliënten koppelen aan dichtstbijzijnde graph-knoop
3. Reistijdenmatrix berekenen (Dijkstra voor alle relevante knooppunten)
4. OR-Tools VRP oplossen met capaciteitsbeperking (5 stops/medewerker)
5. Routes visualiseren op Folium-kaart met 20 unieke kleuren

## 1. Bibliotheken importeren

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
import folium
from scipy.spatial import cKDTree
from ortools.constraint_solver import routing_enums_pb2, pywrapcp
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

## 2. Wegennet laden en graph bouwen

In [ ]:
edges_df = pd.read_csv('../output/heerlen_edge_table.csv')
print(f'Edges loaded: {len(edges_df)}')
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

G          = nx.Graph()
node_coords = {}  # node_id -> (lon, lat)
edge_geom   = {}  # (u, v) -> geometry (both directions)

for _, row in edges_df.iterrows():
    geom   = row['geometry']
    coords = list(geom.coords)
    u, v   = row['u'], row['v']
    G.add_edge(u, v, weight=row['travel_time_min'], geometry=geom)
    node_coords[u] = (coords[0][0],  coords[0][1])
    node_coords[v] = (coords[-1][0], coords[-1][1])
    edge_geom[(u, v)] = geom
    edge_geom[(v, u)] = geom

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.')

# k-d tree for fast nearest-node lookup
node_ids      = list(node_coords.keys())
node_lons_arr = np.array([node_coords[n][0] for n in node_ids])
node_lats_arr = np.array([node_coords[n][1] for n in node_ids])
kd_tree = cKDTree(np.column_stack((node_lons_arr, node_lats_arr)))

def nearest_node(lon, lat):
    _, idx = kd_tree.query([lon, lat])
    return node_ids[idx]

## 3. Medewerkers en cliënten laden

In [ ]:
# ── Employees (manually geocoded Heerlen addresses) ───────────────────────
employee_data = [
    ('employees 1',  50.8872, 5.9812),
    ('employees 2',  50.8895, 5.9820),
    ('employees 3',  50.8883, 5.9830),
    ('employees 4',  50.8855, 5.9795),
    ('employees 5',  50.8945, 5.9660),
    ('employees 6',  50.8878, 5.9808),
    ('employees 7',  50.8948, 5.9700),
    ('employees 8',  50.8870, 5.9825),
    ('employees 9',  50.8868, 5.9817),
    ('employees 10', 50.8785, 5.9750),
    ('employees 11', 50.8840, 5.9810),
    ('employees 12', 50.8860, 5.9835),
    ('employees 13', 50.8850, 5.9880),
    ('employees 14', 50.8890, 5.9822),
    ('employees 15', 50.8710, 5.9920),
    ('employees 16', 50.8810, 5.9680),
    ('employees 17', 50.8952, 5.9672),
    ('employees 18', 50.8875, 5.9805),
    ('employees 19', 50.8940, 5.9665),
    ('employees 20', 50.8790, 5.9760),
]
employees_df = pd.DataFrame(employee_data, columns=['name', 'lat', 'lon'])
employees_df['node'] = employees_df.apply(
    lambda r: nearest_node(r['lon'], r['lat']), axis=1
)
print(f'Employees: {len(employees_df)}')

# ── Clients from clients.csv ──────────────────────────────────────────────
clients_df = pd.read_csv('../output/clients.csv')
print('Client columns:', clients_df.columns.tolist())

# Auto-detect coordinate column
coord_col = None
for col in clients_df.columns:
    sample = clients_df[col].dropna().astype(str).iloc[0]
    parts  = sample.replace(',', ' ').replace(';', ' ').split()
    if len(parts) == 2:
        try:
            float(parts[0]); float(parts[1])
            coord_col = col
            break
        except ValueError:
            pass

coord_col = coord_col or clients_df.columns[0]
print(f'Coordinate column: "{coord_col}"')

def split_coords(s):
    parts = str(s).replace(';', ' ').replace(',', ' ').split()
    return (float(parts[0]), float(parts[1])) if len(parts) == 2 else (np.nan, np.nan)

clients_df[['lat', 'lon']] = clients_df[coord_col].apply(
    lambda x: pd.Series(split_coords(x))
)
clients_df = clients_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
clients_df['client_id'] = clients_df.index
clients_df['node'] = clients_df.apply(
    lambda r: nearest_node(r['lon'], r['lat']), axis=1
)
print(f'Clients: {len(clients_df)}')

## 4. Reistijdenmatrix berekenen

We berekenen de kortste reistijd (Dijkstra) vanuit elk thuisknooppunt en elk cliëntknooppunt.  
De resulterende matrix heeft dimensie **(20 depots + 100 cliënten) × (20 depots + 100 cliënten)**.

In [ ]:
N_EMPLOYEES = len(employees_df)   # 20
N_CLIENTS   = len(clients_df)     # 100

# Index layout for OR-Tools:
#   0 .. 19  → employee home nodes (depot per vehicle)
#  20 .. 119 → client nodes
all_nodes = (
    employees_df['node'].tolist() +
    clients_df['node'].tolist()
)  # length = 120

unique_sources = list(set(all_nodes))
print(f'Running Dijkstra from {len(unique_sources)} unique graph nodes ...')

dist_from = {}  # graph_node -> {graph_node: minutes}
for i, src in enumerate(unique_sources):
    dist_from[src] = nx.single_source_dijkstra_path_length(
        G, src, weight='weight'
    )
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(unique_sources)} done')

print('Building travel-time matrix ...')

N_TOTAL = N_EMPLOYEES + N_CLIENTS  # 120

# Build integer matrix (OR-Tools requires int)
# Scale minutes × 100 to keep 2 decimal precision as integers
SCALE = 100
time_matrix = np.zeros((N_TOTAL, N_TOTAL), dtype=np.int64)

for i in range(N_TOTAL):
    src_graph_node = all_nodes[i]
    lengths        = dist_from[src_graph_node]
    for j in range(N_TOTAL):
        dst_graph_node = all_nodes[j]
        t = lengths.get(dst_graph_node, float('inf'))
        time_matrix[i][j] = int(t * SCALE) if t != float('inf') else 10_000_000

print(f'Time matrix shape: {time_matrix.shape}')
print(f'Min travel time: {time_matrix[time_matrix > 0].min() / SCALE:.2f} min')
print(f'Max finite travel time: {time_matrix[time_matrix < 10_000_000].max() / SCALE:.2f} min')

## 5. OR-Tools VRP oplossen

We formuleren het als een **Capacitated VRP met meerdere depots**:
- Elk voertuig (medewerker) heeft zijn eigen depot (thuisadres)
- Elke cliënt heeft een demand = 1
- Capaciteit per voertuig = 5
- Doelstelling: minimaliseer de totale reistijd
- Oplosser: **PATH_CHEAPEST_ARC** start + **GUIDED_LOCAL_SEARCH** metaheuristiek

In [ ]:
CAPACITY = 5  # clients per employee

# ── OR-Tools data model ───────────────────────────────────────────────────
def create_data_model():
    data = {}
    data['time_matrix']  = time_matrix.tolist()
    data['num_vehicles'] = N_EMPLOYEES
    # Each vehicle starts AND ends at its own home (indices 0-19)
    data['starts'] = list(range(N_EMPLOYEES))
    data['ends']   = list(range(N_EMPLOYEES))
    # Demand: employees (depots) have demand 0, clients have demand 1
    data['demands']    = [0] * N_EMPLOYEES + [1] * N_CLIENTS
    data['capacities'] = [CAPACITY] * N_EMPLOYEES
    return data

data = create_data_model()

# ── Create routing model ──────────────────────────────────────────────────
manager = pywrapcp.RoutingIndexManager(
    len(data['time_matrix']),
    data['num_vehicles'],
    data['starts'],
    data['ends']
)
routing = pywrapcp.RoutingModel(manager)

# ── Travel time callback ──────────────────────────────────────────────────
def time_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node   = manager.IndexToNode(to_index)
    return data['time_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(time_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ── Capacity constraint ───────────────────────────────────────────────────
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,                       # no slack
    data['capacities'],      # max 5 per vehicle
    True,                    # start cumul at zero
    'Capacity'
)

# ── Solver parameters ─────────────────────────────────────────────────────
search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_params.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_params.time_limit.seconds = 30   # optimise for 30 seconds
search_params.log_search = True

print('Solving VRP ...')
solution = routing.SolveWithParameters(search_params)

if solution:
    print(f'Solution found! Total cost: {solution.ObjectiveValue() / SCALE:.1f} min')
else:
    print('No solution found!')

## 6. Oplossing extraheren

In [ ]:
def extract_routes(solution, routing, manager, data):
    """
    Extract routes from OR-Tools solution.
    Returns list of dicts, one per vehicle:
      {
        'vehicle_id':   int,
        'node_indices': [int, ...],   # indices into all_nodes (0-119)
        'client_ids':   [int, ...],   # original client IDs (0-99)
        'total_time':   float,        # minutes
      }
    """
    routes = []
    for vehicle_id in range(data['num_vehicles']):
        index       = routing.Start(vehicle_id)
        node_indices = []
        route_time   = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            node_indices.append(node)
            prev_index = index
            index      = solution.Value(routing.NextVar(index))
            route_time += routing.GetArcCostForVehicle(
                prev_index, index, vehicle_id
            )

        node_indices.append(manager.IndexToNode(index))  # end depot

        # Client indices are N_EMPLOYEES..N_TOTAL-1 in the matrix
        client_ids = [
            ni - N_EMPLOYEES
            for ni in node_indices
            if N_EMPLOYEES <= ni < N_TOTAL
        ]

        routes.append({
            'vehicle_id':   vehicle_id,
            'node_indices': node_indices,
            'client_ids':   client_ids,
            'total_time':   route_time / SCALE,
        })
    return routes


routes = extract_routes(solution, routing, manager, data)

print('=== Route samenvatting ===')
total_minutes = 0
for r in routes:
    emp  = employees_df.loc[r['vehicle_id'], 'name']
    n    = len(r['client_ids'])
    t    = r['total_time']
    total_minutes += t
    print(f'{emp:15s}: {n} cliënten | {t:5.1f} min | stops: {r["client_ids"]}')

print(f'\nTotale reistijd alle medewerkers: {total_minutes:.1f} min')
print(f'Gemiddeld per medewerker:          {total_minutes/N_EMPLOYEES:.1f} min')

## 7. Kaart bouwen

Elke medewerker krijgt een unieke kleur. Routes volgen de echte wegen uit het wegennet.  
Thuisadressen worden als grote gekleurde cirkels getoond, cliënten als kleine cirkels.

In [ ]:
EMPLOYEE_COLORS = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#469990', '#dcbeff', '#9A6324', '#ff8c00', '#800000',
    '#aaffc3', '#808000', '#00bfff', '#000075', '#808080',
]

def nodes_to_latlon(path_nodes):
    """
    Walk a sequence of graph node IDs and collect road-following (lat, lon)
    coordinates by reading each edge's full WKT geometry.
    Handles reversed geometries automatically.
    """
    latlon = []
    for i in range(len(path_nodes) - 1):
        u, v = path_nodes[i], path_nodes[i + 1]
        geom = edge_geom.get((u, v))
        if geom is None:
            # Fallback: straight line between node coordinates
            cu, cv = node_coords.get(u), node_coords.get(v)
            if cu:
                latlon.append((cu[1], cu[0]))
            if cv:
                latlon.append((cv[1], cv[0]))
            continue
        coords = list(geom.coords)
        # Flip geometry if it runs in the wrong direction
        cu = node_coords.get(u)
        if cu and len(coords) >= 2:
            if abs(coords[-1][0] - cu[0]) < abs(coords[0][0] - cu[0]):
                coords = coords[::-1]
        latlon.extend([(lat, lon) for lon, lat in coords])
    return latlon


def road_segment(graph_node_a, graph_node_b):
    """
    Compute the shortest path between two graph nodes and return
    the full road-following (lat, lon) coordinate list.
    """
    try:
        path = nx.shortest_path(G, source=graph_node_a,
                                target=graph_node_b, weight='weight')
        return nodes_to_latlon(path)
    except nx.NetworkXNoPath:
        ca, cb = node_coords.get(graph_node_a), node_coords.get(graph_node_b)
        result = []
        if ca:
            result.append((ca[1], ca[0]))
        if cb:
            result.append((cb[1], cb[0]))
        return result


# ── Build map ─────────────────────────────────────────────────────────────
center_lat = float(np.mean(node_lats_arr))
center_lon = float(np.mean(node_lons_arr))

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    tiles='CartoDB positron'
)

# Road network background
for _, row in edges_df.iterrows():
    latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
    folium.PolyLine(
        locations=latlon, color='#cccccc', weight=1, opacity=0.3
    ).add_to(m)

print('Drawing routes ...')
for route in routes:
    vid    = route['vehicle_id']
    color  = EMPLOYEE_COLORS[vid % len(EMPLOYEE_COLORS)]
    emp    = employees_df.loc[vid]
    ni_seq = route['node_indices']  # indices into all_nodes

    # Convert matrix indices to graph nodes
    graph_seq = [all_nodes[ni] for ni in ni_seq]

    stop_labels = (['thuis'] +
                   [f'stop {k}' for k in range(1, len(route['client_ids']) + 1)] +
                   ['thuis'])

    for seg_i in range(len(graph_seq) - 1):
        latlon = road_segment(graph_seq[seg_i], graph_seq[seg_i + 1])
        if len(latlon) >= 2:
            folium.PolyLine(
                locations=latlon,
                color=color,
                weight=4,
                opacity=0.85,
                tooltip=(
                    f"{emp['name']} | "
                    f"{stop_labels[seg_i]} → {stop_labels[min(seg_i+1, len(stop_labels)-1)]}"
                )
            ).add_to(m)

# Home markers
for vid, emp in employees_df.iterrows():
    color = EMPLOYEE_COLORS[vid % len(EMPLOYEE_COLORS)]
    r     = routes[vid]
    folium.Marker(
        location=[emp['lat'], emp['lon']],
        icon=folium.DivIcon(
            html=(
                f'<div style="width:22px;height:22px;background:{color};'
                'border:3px solid white;border-radius:50%;'
                'box-shadow:0 2px 6px rgba(0,0,0,.5);"></div>'
            ),
            icon_size=(22, 22),
            icon_anchor=(11, 11)
        ),
        popup=folium.Popup(
            f"<b>{emp['name']}</b><br>"
            f"Totale reistijd: {r['total_time']:.1f} min<br>"
            f"Cliënten: {r['client_ids']}",
            max_width=240
        ),
        tooltip=f"{emp['name']} (thuis)"
    ).add_to(m)

# Client markers — colored by assigned employee
client_info = {}   # client_id -> (color, emp_name, stop_num)
for route in routes:
    vid   = route['vehicle_id']
    color = EMPLOYEE_COLORS[vid % len(EMPLOYEE_COLORS)]
    name  = employees_df.loc[vid, 'name']
    for stop_num, cid in enumerate(route['client_ids'], start=1):
        client_info[cid] = (color, name, stop_num)

for _, client in clients_df.iterrows():
    cid = int(client['client_id'])
    if cid not in client_info:
        continue
    color, emp_name, stop_num = client_info[cid]
    folium.CircleMarker(
        location=[client['lat'], client['lon']],
        radius=7,
        color='white',
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(
            f"<b>Cliënt {cid}</b><br>"
            f"Medewerker: {emp_name}<br>"
            f"Stop #{stop_num}",
            max_width=180
        ),
        tooltip=f"Cliënt {cid} | {emp_name} | stop {stop_num}"
    ).add_to(m)

print('Map drawn successfully.')

## 8. Legenda toevoegen en kaart opslaan

In [ ]:
legend_rows = ''
for route in routes:
    vid   = route['vehicle_id']
    color = EMPLOYEE_COLORS[vid % len(EMPLOYEE_COLORS)]
    name  = employees_df.loc[vid, 'name']
    t     = route['total_time']
    cids  = route['client_ids']
    legend_rows += (
        '<tr>'
        f'<td><div style="width:14px;height:14px;background:{color};'
        'border:1px solid #ccc;border-radius:3px;margin:2px;"></div></td>'
        f'<td style="padding:1px 8px;white-space:nowrap;"><b>{name}</b></td>'
        f'<td style="color:#444;white-space:nowrap;">{t:.0f} min &nbsp; '
        f'cliënten: {cids}</td>'
        '</tr>'
    )

legend_html = (
    '<div style="position:fixed;bottom:20px;left:20px;z-index:1000;'
    'background:rgba(255,255,255,0.96);padding:12px 16px;border-radius:8px;'
    'font-size:11px;font-family:sans-serif;'
    'box-shadow:0 2px 12px rgba(0,0,0,.3);'
    'max-height:500px;overflow-y:auto;">'
    '<b style="font-size:13px;">OR-Tools VRP — Routeoverzicht</b><br>'
    '<span style="color:#777;font-size:10px;">'
    '20 medewerkers · 100 cliënten · 5 stops/medewerker</span>'
    '<table style="border-collapse:collapse;margin-top:8px;line-height:1.7;">'
    '<tr>'
    '<th style="text-align:left;"></th>'
    '<th style="text-align:left;padding-right:8px;">Medewerker</th>'
    '<th style="text-align:left;">Reistijd &amp; cliënten</th>'
    '</tr>'
    + legend_rows +
    '</table>'
    '<hr style="margin:8px 0;border-color:#eee;">'
    '<span style="color:#666;">'
    '&#9679; groot = thuisadres &nbsp; &#9679; klein = cliënt</span>'
    '</div>'
)

m.get_root().html.add_child(folium.Element(legend_html))

output_path = '../output/vrp_routes_map.html'
m.save(output_path)
print(f'Map saved: {output_path}')

from IPython.display import IFrame, display
display(IFrame(src='vrp_routes_map.html', width='100%', height=700))

## 9. Gedetailleerde routesamenvatting

In [ ]:
print('=== Gedetailleerde routesamenvatting ===')
rows = []
for route in routes:
    vid = route['vehicle_id']
    rows.append({
        'Medewerker':        employees_df.loc[vid, 'name'],
        'Aantal stops':      len(route['client_ids']),
        'Cliënt-IDs':        str(route['client_ids']),
        'Totale tijd (min)': round(route['total_time'], 1),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

total = summary['Totale tijd (min)'].sum()
avg   = summary['Totale tijd (min)'].mean()
print(f'\nTotale reistijd:  {total:.1f} min')
print(f'Gemiddeld:        {avg:.1f} min/medewerker')
print(f'Langste route:    {summary["Totale tijd (min)"].max():.1f} min')
print(f'Kortste route:    {summary["Totale tijd (min)"].min():.1f} min')

## Planning per medewerker
Voor elke medewerker wordt een gedetailleerde dagplanning gegenereerd:
- **Vertrektijd** vanuit thuisadres
- **Aankomsttijd** bij elke cliënt
- **Zorgtijd** (op basis van `care_hours` uit het bestand)
- **Vertrektijd** naar de volgende stop
- **Thuiskomsttijd** aan het einde van de dag

In [ ]:
from datetime import datetime, timedelta

# ─── Startmoment van de werkdag ───────────────────────────────────────────────
START_OF_DAY = datetime.strptime("08:00", "%H:%M")

def minutes_to_time(base: datetime, minutes: float) -> str:
    return (base + timedelta(minutes=minutes)).strftime("%H:%M")

def build_schedule(route: dict, employees_df, clients_df, time_matrix, SCALE) -> list[dict]:
    """
    Bouw een stap-voor-stap dagplanning voor één medewerker.
    Retourneert een lijst van dictionaries per stop.
    """
    vid        = route["vehicle_id"]
    client_ids = route["client_ids"]
    ni_seq     = route["node_indices"]   # indices in time_matrix (0-119)

    rows = []
    current_time = 0.0  # minuten na START_OF_DAY

    emp_name = employees_df.loc[vid, "name"]

    # Eerste rij: vertrek vanuit huis
    rows.append({
        "Stop":          "Thuis (vertrek)",
        "Cliënt":        "—",
        "Aankomsttijd":  "—",
        "Zorgtijd (min)": "—",
        "Vertrektijd":   minutes_to_time(START_OF_DAY, current_time),
    })

    for step, cid in enumerate(client_ids):
        # Reistijd van vorige stop naar deze cliënt
        from_ni = ni_seq[step]       # matrix-index van vorige stop (start = depot)
        to_ni   = ni_seq[step + 1]   # matrix-index van deze cliënt
        travel  = time_matrix[from_ni][to_ni] / SCALE  # minuten

        current_time += travel
        arrival_time  = minutes_to_time(START_OF_DAY, current_time)

        # Zorgtijd ophalen uit clients_df (care_hours → minuten)
        care_hours = clients_df.loc[cid, "care_hours"] if "care_hours" in clients_df.columns else 1.0
        care_min   = float(care_hours) * 60.0
        current_time += care_min
        departure_time = minutes_to_time(START_OF_DAY, current_time)

        client_name = clients_df.loc[cid, "name"] if "name" in clients_df.columns else f"Cliënt {cid}"

        rows.append({
            "Stop":           f"Stop {step + 1}",
            "Cliënt":         client_name,
            "Aankomsttijd":   arrival_time,
            "Zorgtijd (min)": f"{int(care_min)} min",
            "Vertrektijd":    departure_time,
        })

    # Laatste reistijd: terug naar huis
    if client_ids:
        last_ni = ni_seq[len(client_ids)]    # index van laatste cliënt in matrix
        home_ni = ni_seq[len(client_ids) + 1] if len(ni_seq) > len(client_ids) + 1 else ni_seq[-1]
        travel_home = time_matrix[last_ni][home_ni] / SCALE
        current_time += travel_home

    rows.append({
        "Stop":           "Thuis (aankomst)",
        "Cliënt":         "—",
        "Aankomsttijd":   minutes_to_time(START_OF_DAY, current_time),
        "Zorgtijd (min)": "—",
        "Vertrektijd":    "—",
    })

    return rows


# ─── Genereer en toon planning per medewerker ─────────────────────────────────
all_schedules = {}

for route in routes:
    vid      = route["vehicle_id"]
    emp_name = employees_df.loc[vid, "name"]

    if not route["client_ids"]:
        # Medewerker zonder cliënten overslaan
        continue

    schedule_rows = build_schedule(route, employees_df, clients_df, time_matrix, SCALE)
    df_sched = pd.DataFrame(schedule_rows)
    all_schedules[emp_name] = df_sched

    print(f"\n{'═' * 65}")
    print(f"  📋  {emp_name}")
    print(f"{'═' * 65}")
    print(df_sched.to_string(index=False))

print(f"\n\n✅ Planning gegenereerd voor {len(all_schedules)} medewerkers.")

## Export planning naar Excel
Optioneel: exporteer alle planningen als één Excel-bestand met een tabblad per medewerker.

In [ ]:
# Export alle planningen naar één Excel-bestand (één tabblad per medewerker)
output_excel = "../output/dagplanning_medewerkers.xlsx"

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    for emp_name, df_sched in all_schedules.items():
        # Tabblad-naam max 31 tekens (Excel-limiet)
        sheet_name = emp_name[:31]
        df_sched.to_excel(writer, sheet_name=sheet_name, index=False)

        # Maak kolommen wat breder voor leesbaarheid
        ws = writer.sheets[sheet_name]
        for col in ws.columns:
            max_len = max(len(str(cell.value)) for cell in col if cell.value) + 4
            ws.column_dimensions[col[0].column_letter].width = min(max_len, 40)

print(f"✅ Planning opgeslagen: {output_excel}")